# Train Qwen-1.5B baseline (no LoRA)Trains the vanilla Qwen-1.5B on the 5 gold argument mining corpora with full fine tuning (no LoRA). This is an early baseline separate from the four v1-v4 variants published on HuggingFace. Config: `configs/phase2_beta_qwen1.5b.yaml`. The output landed in `models/phase2_student_beta_qwen1.5b/` on the training host.

In [ ]:
%%bash
cd ~/argument-aware-rag && source .venv/bin/activate
git pull origin main

ls phase2_data/unified/
echo ""
# Smoke-test the loader before kicking off the long run
python3 -c "
import sys; sys.path.insert(0, '.')
from src.phase2.dataset import load_unified_corpus
records = load_unified_corpus('phase2_data/unified', exclude_sources=('liararg',))
print(f'Total non-LIARArg: {len(records)}')
" 2>&1 | tail -15

In [ ]:
%%bash
cd ~/argument-aware-rag && source .venv/bin/activate
mkdir -p phase2_student_beta_qwen1.5b
python -m scripts.phase2.train_phase2_beta \
    --config configs/phase2_beta_qwen1.5b.yaml 2>&1 | tee phase2_student_beta_qwen1.5b/train.log

In [ ]:
%%bash
cd ~/argument-aware-rag && source .venv/bin/activate
git pull origin main
# Free any stale GPU allocation from the OOM'd run
python -c "import torch; torch.cuda.empty_cache()" 2>/dev/null
nvidia-smi --query-gpu=memory.used,memory.total --format=csv

# Re-run with the 0.5B config (same path, just different content now)
mkdir -p phase2_student_beta_qwen0.5b
python -m scripts.phase2.train_phase2_beta \
    --config configs/phase2_beta_qwen1.5b.yaml 2>&1 | tee phase2_student_beta_qwen0.5b/train.log